# Ramen: basic model + dataloader setup

In [1]:
import os, sys, random
from types import SimpleNamespace

import numpy as np
import torch
import yaml
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.abspath('../src'))

from models.get_pretrained_model import get_pretrained_model
from datasets import get_dataset_class
from datasets.utils import TaggedMultipleDataset
from methods import get_method_class

In [ ]:
# Same fields main.py's argparse produces
args = SimpleNamespace(
    dataset='CIFAR10C',          # CIFAR10C | CIFAR100C | ImageNetC5K | DomainNet
    model='clip_vitbase32',      # clip_vitbase32 | clip_vitbase16 | clip_vitlarge14 | clip_rn50 | clip_rn101
    tta_algo='Ramen',            # Ramen | Tent | NoAdapt
    tta_mode='mixed',            # single | mixed
    data_root=os.path.expanduser('~/data'),
    batch_size=64,
    num_workers=4,
    seed=0,
    use_tbn=False,
    config_dir='../cfg',
)
args.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
args.max_batch_size = args.batch_size

# Load YAML config: cfg/<dataset>/<algo>.yaml, else cfg/default/<algo>.yaml
args.config = {}
for path in [os.path.join(args.config_dir, args.dataset, f'{args.tta_algo}.yaml'),
             os.path.join(args.config_dir, 'default', f'{args.tta_algo}.yaml')]:
    if os.path.exists(path):
        with open(path) as f:
            args.config = yaml.safe_load(f)
        break
print(args.config)

torch.manual_seed(args.seed); np.random.seed(args.seed); random.seed(args.seed)

{'max_capacity': 7500, 'topk': 50, 'beta': 5.0, 'optimizer': 'signsgd', 'lr': 0.01}


In [3]:
# Pretrained CLIP backbone + its preprocessing transform
model, preprocess = get_pretrained_model(args)

# Multi-domain dataset (one sub-dataset per corruption/domain)
datasets = get_dataset_class(args.dataset)(root=args.data_root, transform=preprocess)
print('environments:', datasets.environments)
print('num_classes:', datasets.num_classes)

# Wrap the backbone with the TTA method (Ramen)
tta_model = get_method_class(args.tta_algo)(model, datasets, args)

environments: ['gaussian_noise', 'shot_noise', 'impulse_noise', 'defocus_blur', 'glass_blur', 'motion_blur', 'zoom_blur', 'snow', 'frost', 'fog', 'brightness', 'contrast', 'elastic_transform', 'pixelate', 'jpeg_compression']
num_classes: 10


In [4]:
# Mixed-domain loader: yields (image, label, domain_idx, sample_idx)
mixed_loader = DataLoader(TaggedMultipleDataset(datasets), batch_size=args.batch_size,
                          shuffle=True, num_workers=args.num_workers, pin_memory=True)

# Single-domain loaders: one per environment, yields (image, label)
single_loaders = {env: DataLoader(ds, batch_size=args.batch_size, shuffle=True,
                                  num_workers=args.num_workers, pin_memory=True)
                  for env, ds in zip(datasets.environments, datasets)}

image, label, domain_idx, sample_idx = next(iter(mixed_loader))
print(image.shape, label.shape, domain_idx[:10])

torch.Size([100, 3, 224, 224]) torch.Size([100]) tensor([ 4,  9,  1, 14,  9,  7,  0,  9, 12,  6])


In [5]:
# Sanity check: one adaptation step on a single batch
image, label = image.to(args.device), label.to(args.device)
logits = tta_model(image)
acc = (logits.argmax(1) == label).float().mean().item()
print(f'batch acc: {acc * 100:.2f}%')

tta_model.reset()  # restore the model / clear Ramen's cache before a real run

batch acc: 73.00%


## Full-dataset evaluation

In [ ]:
from tqdm.auto import tqdm


def run_mixed(tta_model, loader, datasets):
    """All domains shuffled together into one stream; the model keeps adapting throughout."""
    num_correct = torch.zeros(len(datasets), dtype=torch.long)
    num_samples = torch.tensor([len(ds) for ds in datasets])

    for image, label, domain_idx, _ in tqdm(loader):
        image, label = image.to(args.device), label.to(args.device)
        logits = tta_model(image)
        with torch.no_grad():
            correct = (logits.argmax(1) == label).cpu().long()
            num_correct.index_add_(0, domain_idx, correct)

    tta_model.reset()
    return dict(zip(datasets.environments, (num_correct / num_samples).tolist()))


def run_single(tta_model, loaders):
    """Each domain adapted independently; the model is reset after every domain."""
    accs = {}
    for env, loader in loaders.items():
        correct, total = 0, 0
        for image, label in tqdm(loader, desc=env):
            image, label = image.to(args.device), label.to(args.device)
            logits = tta_model(image)
            with torch.no_grad():
                correct += (logits.argmax(1) == label).sum().item()
                total += label.size(0)
        accs[env] = correct / total
        tta_model.reset()
    return accs

In [ ]:
if args.tta_mode == 'mixed':
    accs = run_mixed(tta_model, mixed_loader, datasets)
else:
    accs = run_single(tta_model, single_loaders)

for env, acc in accs.items():
    print(f'{env:>20}: {acc * 100:.2f}%')
print(f'{"average":>20}: {np.mean(list(accs.values())) * 100:.2f}%')